# Target: ClinVar x AlphaFold (exploration)

This notebook is target-agnostic. It expects run outputs produced by `avi run` (or the underlying scripts) and locates files via `PIPELINE_CONFIG_PATH` / `PIPELINE_OUTPUT_DIR`.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd

BASE = Path("..").resolve()

# Support run directories created by `py -m avi init` + `py -m avi run --run-dir ...`.
cfg_path = Path(os.environ.get("PIPELINE_CONFIG_PATH") or (BASE / "pipeline_config.json")).resolve()
data_root = Path(os.environ.get("PIPELINE_OUTPUT_DIR") or (BASE / "data")).resolve()

with open(cfg_path, encoding="utf-8") as f:
    cfg = json.load(f)

basename = cfg.get("output_basename", "target")
uid = cfg.get("uniprot_id", "UNKNOWN")
frag = cfg.get("alphafold_fragment", "F1")

csv_path = data_root / "processed" / f"{basename}_variants_basic.csv"
mappable_path = data_root / "processed" / f"{basename}_missense_mappable.csv"
pdb_path = data_root / "raw" / f"AF-{uid}-{frag}-alphafold.pdb"

print("Config:", cfg_path)
print("Data root:", data_root)
print("Variants CSV:", csv_path, "exists:", csv_path.is_file())
print("AlphaFold PDB:", pdb_path, "exists:", pdb_path.is_file())

df = pd.read_csv(csv_path)
df.head()


## Missense subset (if available)

In [ ]:
if mappable_path.is_file():
    sub = pd.read_csv(mappable_path)
    sub.head()
else:
    print("Missing missense subset:", mappable_path)


## Summaries (all variants)

Counts by coarse clinical bucket and a quick view of high-signal missense rows when the subset CSV exists.

In [ ]:
bucket_col = "clinical_significance_bucket" if "clinical_significance_bucket" in df.columns else None
if bucket_col:
    print(df[bucket_col].value_counts().rename_axis("bucket").reset_index(name="count").to_string(index=False))
else:
    print("No clinical_significance_bucket column — re-run process_tp53_variants.py / avi run.")

if mappable_path.is_file():
    sub_sum = pd.read_csv(mappable_path)
    print("Missense mappable rows:", len(sub_sum))
    if "clinical_significance_bucket" in sub_sum.columns and "alphafold_confidence_score" in sub_sum.columns:
        med = (
            sub_sum.groupby("clinical_significance_bucket", dropna=False)["alphafold_confidence_score"]
            .median()
            .rename("median_plddt")
            .reset_index()
        )
        print(med.to_string(index=False))

## Minimal evaluation: pathogenic vs benign on structure-backed features

**Hypothesis (exploratory):** missense variants labeled pathogenic or likely pathogenic may differ in median AlphaFold pLDDT (local model confidence) or in CA distance to the protein centroid (a crude burial proxy) compared with benign or likely benign.

This is not a clinical classifier — it is a reproducible sanity check you can extend with better solvent-accessibility estimates, domain annotations, or curated hotspots.

In [ ]:
import matplotlib.pyplot as plt

if not mappable_path.is_file():
    print("Skip evaluation: missense mappable CSV not found.")
else:
    _hdr = pd.read_csv(mappable_path, nrows=0)
    if "clinical_significance_bucket" not in _hdr.columns:
        print("Re-run the pipeline to refresh CSVs with clinical_significance_bucket.")
    else:
        ev = pd.read_csv(mappable_path)
        b = ev["clinical_significance_bucket"]
        harm_mask = b.isin(["Pathogenic", "Likely pathogenic"])
        ben_mask = b.isin(["Benign", "Likely benign"])
        harm = ev[harm_mask]
        ben = ev[ben_mask]
        print("Pathogenic + likely pathogenic (missense, in structure):", len(harm))
        print("Benign + likely benign (missense, in structure):", len(ben))

        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        for ax, col, title in (
            (axes[0], "alphafold_confidence_score", "Per-residue pLDDT"),
            (axes[1], "ca_distance_to_centroid_angstrom", "CA distance to centroid (Å)"),
        ):
            if col not in ev.columns or harm[col].notna().sum() == 0 or ben[col].notna().sum() == 0:
                ax.set_title(title + " (insufficient data)")
                ax.axis("off")
                continue
            data = [harm[col].dropna(), ben[col].dropna()]
            ax.boxplot(data, labels=["Path / LP", "Ben / LB"])
            ax.set_title(title)
            ax.set_ylabel(col)
        plt.tight_layout()
        plt.show()

## Config-defined analysis region (paper-style slice)

When `pipeline_config.json` (or a preset) sets `analysis_region_start` / `analysis_region_end` / `analysis_region_label` — for example the TP53 DNA-binding domain — filter missense variants by **ClinVar protein position** and compare pathogenic vs benign within that window.

In [ ]:
import matplotlib.pyplot as plt

ra, rb = cfg.get("analysis_region_start"), cfg.get("analysis_region_end")
lab = cfg.get("analysis_region_label")
if not mappable_path.is_file() or ra is None or rb is None:
    print("Skip region plot: no missense CSV or analysis_region_* not set in config.")
else:
    ev = pd.read_csv(mappable_path)
    pos = pd.to_numeric(ev["protein_position"], errors="coerce")
    core = ev[(pos >= int(ra)) & (pos <= int(rb))].copy()
    print((lab or "Region") + f": residues {int(ra)}–{int(rb)}, n={len(core)}")
    b = core["clinical_significance_bucket"]
    harm = core[b.isin(["Pathogenic", "Likely pathogenic"])]
    ben = core[b.isin(["Benign", "Likely benign"])]
    print("  Path/LP:", len(harm), "  Ben/LB:", len(ben))
    cols = [c for c in ("alphafold_confidence_score", "residue_sasa_angstrom2") if c in core.columns]
    if not cols or len(harm) < 3 or len(ben) < 3:
        print("Insufficient data for region boxplots.")
    else:
        fig, axes = plt.subplots(1, len(cols), figsize=(4 * len(cols), 4))
        if len(cols) == 1:
            axes = [axes]
        for ax, col in zip(axes, cols):
            data = [harm[col].dropna(), ben[col].dropna()]
            ax.boxplot(data, labels=["Path / LP", "Ben / LB"])
            ax.set_title(col)
        plt.suptitle(lab or f"Region {ra}-{rb}")
        plt.tight_layout()
        plt.show()